# Task 1: Data handling and pre-processing

This notebook section implements only preprocessing and train/validation splitting.


In [ ]:
# Import libraries for data handling
from pathlib import Path
from typing import Tuple, Dict

import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
# Set the expected class names from the project description
EXPECTED_CLASSES: list[str] = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
]

# Allow only common image extensions
VALID_IMAGE_EXTENSIONS: set[str] = {".jpg", ".jpeg", ".png"}


def preprocess(data_folder: str) -> tuple[pd.DataFrame, dict[int, str]]:
    """
    Build a clean dataframe from class folders.

    Returns:
    - DataFrame with columns: folder, file_name, label
    - Dictionary mapping numeric label -> class name
    """

    # Convert input path string to Path object
    data_path = Path(data_folder)

    # Check 1: data folder must exist
    if not data_path.exists() or not data_path.is_dir():
        raise FileNotFoundError(f"Data folder does not exist: {data_path}")

    # Sort expected classes so label assignment is deterministic
    sorted_classes = sorted(EXPECTED_CLASSES)

    # Check 2: all expected class folders must exist
    missing_classes = [c for c in sorted_classes if not (data_path / c).is_dir()]
    if missing_classes:
        raise ValueError(f"Missing class folders: {missing_classes}")

    # Build class <-> label maps with stable ordering
    class_to_label: dict[str, int] = {class_name: idx for idx, class_name in enumerate(sorted_classes)}
    label_to_class: dict[int, str] = {idx: class_name for class_name, idx in class_to_label.items()}

    # Collect rows here
    rows: list[dict[str, object]] = []

    # Read all image files class-by-class
    for class_name in sorted_classes:
        class_folder = data_path / class_name
        label = class_to_label[class_name]

        # Sort files for deterministic row order
        for file_path in sorted(class_folder.iterdir()):
            # Ignore hidden files like .DS_Store
            if file_path.name.startswith('.'):
                continue

            # Keep only valid image files
            if file_path.is_file() and file_path.suffix.lower() in VALID_IMAGE_EXTENSIONS:
                rows.append(
                    {
                        "folder": str(class_folder),
                        "file_name": file_path.name,
                        "label": label,
                    }
                )

    # Create dataframe with exact required columns
    df = pd.DataFrame(rows, columns=["folder", "file_name", "label"])

    # Check 3: dataframe must not be empty
    if df.empty:
        raise ValueError("No valid image files found in dataset.")

    # Check 4: every label must be valid
    valid_labels = set(label_to_class.keys())
    if not df["label"].isin(valid_labels).all():
        raise ValueError("Found invalid labels in dataframe.")

    return df, label_to_class


def split_data(
    df: pd.DataFrame,
    validation_size: float = 0.2,
    random_state: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split dataframe into stratified train and validation sets.

    - Random split using train_test_split
    - Stratified by label column
    - Reproducible with fixed random_state
    """

    # Basic safety checks for input dataframe
    required_cols = {"folder", "file_name", "label"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")

    if df.empty:
        raise ValueError("Input dataframe is empty.")

    # Run stratified split
    train_df, val_df = train_test_split(
        df,
        test_size=validation_size,
        random_state=random_state,
        stratify=df["label"],
        shuffle=True,
    )

    # Reset indices as requested
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    return train_df, val_df


In [ ]:
# Example usage block (Task 1 only)
DATA_FOLDER = "data"

df, label_map = preprocess(DATA_FOLDER)
train_df, val_df = split_data(df)

print(df.head())
print(label_map)
print(train_df["label"].value_counts().sort_index())
print(val_df["label"].value_counts().sort_index())
